## Topic 5: JSON Loading

### Concept, Intuition, Why It Exists

- Target: `related_documents_json/*.json` — output of an external conversion/OCR team's pipeline, one JSON file per source PDF, one entry per page inside it.
- **Why this loader exists instead of an in-house PDF parser**: parsing PDFs well (tables, scanned pages, OCR, encrypted files) is a genuinely hard, specialized problem — hard enough that a separate team owns it centrally. Consuming their JSON means there's exactly one source of truth for "what does this PDF actually say," instead of two independent PDF-parsing code paths that could silently disagree on the same source file.
- This loader's job is narrow and deliberate: read what the conversion team produced and map it into Documents. It does **not** judge OCR quality, confidence thresholds, or scanned-page detection — that responsibility stays with the team that owns the conversion pipeline.

### Internal Working

1. Read and parse the JSON file.
2. For each entry in the document's page list, build one Document: the page's text becomes `page_content`; `metadata` carries source filename, document code, version, and page number — exactly the provenance needed to trace a retrieved chunk back to a specific page of a specific document version.
3. A folder-level loader walks every `.json` file in a directory and concatenates their Documents — the entry point a scheduled ingestion job would actually call.

### How It's Implemented In This Project

- Loading one JSON file returns one Document per page.
- Loading the whole folder returns every page across every source document in one flat list, each still traceable to its exact source/page via metadata.
- **What swapping PDFs for JSON proved**: switching the entire upstream source from PDFs-in to JSON-in required changing exactly one file. Nothing downstream — dedup, the incremental manifest, chunking — needed to know anything changed, because the Document shape never changed. The manifest now hashes `.json` files instead of `.pdf` files with zero logic changes, proof it was never PDF-specific in the first place.

### Real-World Issues, Edge Cases, Debugging, Monitoring

- **Trusting upstream content without question is a real, named risk** — if the conversion team's pipeline produces a page with wrong text (e.g. OCR'd the wrong page, or pages swapped during conversion), this loader has no way to catch it, by design. That's the trade-off of not owning quality control: you gain simplicity and a single source of truth, you lose independent verification.
- **Missing/renamed fields**: if the conversion team renames a field or stops emitting one, reading it with a safe default returns `None` silently instead of raising — metadata quietly goes missing across every newly ingested document with no error anywhere. Worth a lightweight assertion on the fields you actually depend on, even without taking on quality judgment.
- **Empty pages**: a page with empty or near-empty text loads as a valid, near-useless Document with no signal that anything is wrong. Since this isn't your responsibility to validate quality-wise, the practical stance is: ingest what's given, and rely on the conversion team's own monitoring to catch extraction failures upstream — not duplicate that effort here.
- **New dependency risk**: this pipeline now depends on another team's pipeline being timely and correct. If their conversion job is delayed, you have stale or missing data with no PDF fallback.

### Design Decisions & Trade-offs

- Gained by consuming JSON instead of parsing PDFs in-house: a simpler pipeline with no OCR/PDF-parsing code to maintain, the benefit of a team that's better at PDF parsing than a one-off loader, and one clear source of truth for "what does this PDF say."
- Given up: independent verification against the source PDF, and a dependency on someone else's pipeline being timely and correct — extraction quality control now sits entirely with the upstream team.
- This is a **separation-of-concerns decision**, not a shortcut. Keeping an in-house parser *and* consuming the conversion team's JSON would mean two independent code paths that could extract different text from the same PDF — a worse outcome than trusting one well-maintained source.

### Alternatives & When To Use Each

- Consume a central conversion team's JSON (this project) — a specialized team already owns PDF/OCR parsing; you want one source of truth and want to stay out of that responsibility.
- In-house PDF parsing — no central conversion pipeline exists; you need full control over extraction timing and are willing to own quality.
- Hybrid fallback — in-house parsing only if the JSON is missing/stale, accepting the two-source-of-truth risk this design avoids, in exchange for resilience.

### Common Mistakes & Production Failures

- Silently swallowing a missing field with a default and never noticing an entire metadata dimension has gone missing across a batch of documents.
- Re-implementing quality checks that duplicate effort the upstream team already owns, blurring a deliberately drawn responsibility boundary.
- Assuming "JSON parsed without error" means "the content is correct" — parsing success and content correctness are different claims.

### Lead-Level Interview Questions

**Q: Why trust an external team's JSON instead of parsing PDFs yourself, when you clearly could?**
A: PDF/OCR parsing is a deep, specialized problem better solved once, centrally, by a team focused on it. Keeping a parallel in-house parser creates two sources of truth for the same PDF that could silently diverge. The trade is accepting a dependency on their pipeline's timeliness and correctness, in exchange for simplicity and not duplicating ownership of a problem that isn't yours to solve.

**Q: This loader doesn't validate OCR confidence or detect scanned pages. Isn't that a gap?**
A: It's a deliberately drawn boundary, not an oversight. Quality control for extraction belongs to the team that owns extraction — duplicating that logic here would mean maintaining two quality bars that can drift apart. The real safeguard is making sure that team's monitoring is solid, not re-implementing their checks downstream.

**Q: A field your metadata depends on gets silently renamed upstream. How would you catch that without taking on full validation responsibility?**
A: A lightweight assertion or schema check on just the fields you actually consume — not a quality judgment on content, just a contract check that the shape you depend on hasn't changed. That's a narrower, fair scope: you're allowed to defend your own contract without taking ownership of someone else's quality bar.

### Hidden Concepts Worth Knowing

- **Data contracts as a first-class artifact**: the JSON shape consumed here is an implicit contract between two teams. Lead-level practice makes contracts explicit (a shared schema, a versioned spec) so breaking changes are caught early, not discovered in production.
- **Drawing responsibility boundaries deliberately**: knowing what *not* to validate is as much a design decision as knowing what to validate — over-owning someone else's quality bar creates duplicated, drifting logic.

### Revision Summary

> JSON loading consumes a conversion team's per-page JSON output as the single source of truth for PDF content, instead of maintaining a competing in-house PDF parser. It maps pages to Documents and carries provenance metadata, but deliberately does not judge extraction quality — that responsibility stays with the team that owns conversion. The Document abstraction is what let this entire upstream source swap (PDF → JSON) touch exactly one file.

In [1]:
"""
json_loader.py
----------------
Loads the conversion/OCR team's JSON output (one JSON file per source
document, one entry per page) as ONE Document per PAGE.

Expected JSON shape (matches the conversion team's contract):
{
  "source_pdf": "02_FD_Product_Guide.pdf",
  "document_code": "FD-PROD-02",
  "version": "1.0",
  "pages": [
    {"page_number": 1, "text": "..."},
    {"page_number": 2, "text": "..."}
  ]
}

Why this loader exists at all: it replaces an in-house PDFLoader. Parsing
PDFs (tables, scanned pages, OCR) is a hard, specialized problem already
solved -- once, centrally -- by the conversion team. Consuming their JSON
means this pipeline has exactly ONE source of truth for "what does this
PDF actually say", instead of two PDF-parsing code paths that could
silently disagree.

NOTE: this loader deliberately does NOT judge OCR/extraction quality
(confidence thresholds, scanned-page detection, etc.) -- that is owned by
the conversion/OCR team, not this pipeline.
"""

import os
import glob
import json
from document import make_document

JSON_FOLDER = "../data/related_documents_json/"


def lazy_load_json(file_path: str):
    """Generator version -- yields one Document per page entry in the JSON."""
    with open(file_path, encoding="utf-8") as f:
        data = json.load(f)

    for page in data.get("pages", []):
        yield make_document(
            page_content=page.get("text", ""),
            metadata={
                "source": data.get("source_pdf", file_path),
                "document_code": data.get("document_code"),
                "version": data.get("version"),
                "page": page.get("page_number"),
            },
        )


def load_json(file_path: str) -> list:
    return list(lazy_load_json(file_path))


def load_all_json(folder: str = JSON_FOLDER) -> list:
    """Loads every .json file in a folder, concatenating all their Documents."""
    documents = []
    for path in sorted(glob.glob(os.path.join(folder, "*.json"))):
        documents.extend(load_json(path))
    return documents


if __name__ == "__main__":
    all_docs = load_all_json()
    print(f"Loaded {len(all_docs)} total pages across all JSON files")
    sources = sorted(set(d["metadata"]["source"] for d in all_docs))
    print(f"  Sources: {sources}")

Loaded 17 total pages across all JSON files
  Sources: ['01_FD_FAQ.pdf', '02_FD_Product_Guide.pdf', '03_FD_SOPs.pdf', '04_FD_Policy_Reference.pdf']
